## **1. 필요한 패키지 import**

In [12]:
import os
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils

%matplotlib inline

In [13]:
# 결과 재현을 위해 시드 고정
manualSeed = 999
random.seed(manualSeed)
torch.manual_seed(manualSeed)

In [14]:
# 실험 재현성을 위한 시드 고정
manualSeed = 999
random.seed(manualSeed)
torch.manual_seed(manualSeed)

# 하이퍼파라미터 설정
workers = 2
batch_size = 128
image_size = 64
nc = 3          # RGB 채널
nz = 100        # latent vector 크기
ngf = 64        # Generator feature map 크기
ndf = 64        # Discriminator feature map 크기
num_epochs = 5
lr = 0.0002
beta1 = 0.5
ngpu = 1

# device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 중인 device:", device)

사용 중인 device: cuda


## **2. 데이터셋 로드**

- CelebA Dataset: 유명인 얼굴 이미지로 구성된 공개 데이터셋.대량의 얼굴 이미지 데이터를 이용해서
  - 얼굴 생성 실험
  - 얼굴 속성(attribute) 분석
  - GAN과 같은 생성모델 성능 평가
- 를 수행할 수 있도록 만든 데이터셋

In [15]:
# gdown 설치
!pip -q install gdown

# CelebA aligned images zip 다운로드
!gdown --id 0B7EVK8r0v71pZjFTYXZWM3FlRnM -O /content/img_align_celeba.zip

# 압축 해제
!unzip -q /content/img_align_celeba.zip -d /content/

# 압축 해제 결과 확인
!ls /content | head
!ls /content/img_align_celeba | head

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM
From (redirected): https://drive.google.com/uc?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM&confirm=t&uuid=45413e11-530a-44c5-ad12-3fd0b130f20a
To: /content/img_align_celeba.zip
100% 1.44G/1.44G [00:13<00:00, 103MB/s]
celeba
img_align_celeba
img_align_celeba.zip
sample_data
000001.jpg
000002.jpg
000003.jpg
000004.jpg
000005.jpg
000006.jpg
000007.jpg
000008.jpg
000009.jpg
000010.jpg


In [16]:
import os
import shutil

src = "/content/img_align_celeba"
dst_root = "/content/celeba"
dst = os.path.join(dst_root, "faces")

os.makedirs(dst, exist_ok=True)

# 이미지 파일을 ImageFolder 형식에 맞게 faces 하위 폴더로 이동
for fname in os.listdir(src):
    shutil.move(os.path.join(src, fname), os.path.join(dst, fname))

print("done")
print(os.listdir(dst_root))
print(len(os.listdir(dst)))

done
['faces']
202599


## **3. Generator 정의**

In [18]:
# Generator는 랜덤 노이즈를 입력받아 가짜 이미지를 생성한다.
class Generator(nn.Module):
    def __init__(self, ngpu):
        super(Generator, self).__init__()
        self.ngpu = ngpu

        self.main = nn.Sequential(
            # 입력: Z latent vector
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            # feature map 크기 확장
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # 최종 출력: 64x64 RGB 이미지
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

In [19]:
# 코랩 환경에서 GPU 사용 가능 여부를 확인하여 device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 중인 device:", device)

사용 중인 device: cuda


In [ ]:
# Generator 모델 생성
netG = Generator(ngpu).to(device)

# 가중치 초기화 적용
netG.apply(weights_init)

# 모델 구조 출력
print(netG)

# Generator는 noise vector를 입력받아 얼굴 이미지를 생성하는 역할을 한다.

In [ ]:
# Discriminator 정의: 입력 이미지가 진짜인지 가짜인지 판별
class Discriminator(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu

        self.main = nn.Sequential(
            # 입력: 64x64 RGB 이미지
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # 최종 출력: 진짜일 확률
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
# Discriminator 모델 생성
netD = Discriminator(ngpu).to(device)

# 가중치 초기화 적용
netD.apply(weights_init)

# 모델 구조 출력
print(netD)

# Discriminator는 입력 이미지가 실제 이미지인지 생성된 이미지인지 판별하는 역할

## **4. Loss, Optimizer 설정**

In [ ]:
# Binary Cross Entropy loss 사용
criterion = nn.BCELoss()

# 학습 과정을 비교하기 위한 고정 노이즈
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# real / fake label 값 설정
real_label = 1.
fake_label = 0.

# Adam optimizer 설정
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

## **5. Train**
- 먼저 Discriminator를 학습하여 real과 fake를 구분하도록 한다.
- 이후 Generator를 학습하여 fake 이미지를 real처럼 보이게 만든다.
- 두 모델을 번갈아 업데이트하면서 GAN을 학습한다.

In [ ]:
# 학습 진행 상황 저장용 리스트
img_list = []
G_losses = []
D_losses = []
iters = 0

In [ ]:
print("Starting Training Loop...")

for epoch in range(num_epochs):
    for i, data in enumerate(dataloader, 0):

        ############################
        # (1) Discriminator 업데이트
        ############################
        netD.zero_grad()

        # real image 학습
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), real_label, dtype=torch.float, device=device)

        output = netD(real_cpu).view(-1)
        errD_real = criterion(output, label)
        errD_real.backward()
        D_x = output.mean().item()

        # fake image 생성
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)

        # fake image 학습
        label.fill_(fake_label)
        output = netD(fake.detach()).view(-1)
        errD_fake = criterion(output, label)
        errD_fake.backward()
        D_G_z1 = output.mean().item()

        # Discriminator 업데이트
        errD = errD_real + errD_fake
        optimizerD.step()

        ############################
        # (2) Generator 업데이트
        ############################
        netG.zero_grad()

        # Generator는 fake를 real처럼 보이게 학습
        label.fill_(real_label)
        output = netD(fake).view(-1)
        errG = criterion(output, label)
        errG.backward()
        D_G_z2 = output.mean().item()

        # Generator 업데이트
        optimizerG.step()

        # loss 저장
        G_losses.append(errG.item())
        D_losses.append(errD.item())

        # 중간 결과 출력
        if i % 50 == 0:
            print(f'[{epoch+1}/{num_epochs}][{i}/{len(dataloader)}] '
                  f'Loss_D: {errD.item():.4f} '
                  f'Loss_G: {errG.item():.4f} '
                  f'D(x): {D_x:.4f} '
                  f'D(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f}')

        # 생성 이미지 저장
        if (iters % 500 == 0) or ((epoch == num_epochs - 1) and (i == len(dataloader) - 1)):
            with torch.no_grad():
                fake = netG(fixed_noise).detach().cpu()
            img_list.append(vutils.make_grid(fake, padding=2, normalize=True))

        iters += 1

## **6. Test**

In [ ]:
# 데이터셋 분할, random state는 위에서 사전 정의 완료

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
datasets = {'train': (X_train, y_train), 'val': (X_valid, y_valid)}
dataset_sizes = {'train': len(X_train), 'val': len(X_valid)}

In [ ]:
minmax = ratings.rating.min(), ratings.rating.max()
minmax

In [ ]:
net = EmbeddingNet(
    n_users=n, n_movies=m,
    n_factors=150, hidden=[500, 500, 500],
    embedding_dropout=0.05, dropouts=[0.5, 0.5, 0.25])

In [ ]:
print(n,m)

In [ ]:
lr = 1e-3
wd = 1e-5
bs = 2000
n_epochs = 200
patience = 10
no_improvements = 0
best_loss = np.inf
best_weights = None
history = []
lr_history = []

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

net.to(device)
criterion = nn.MSELoss(reduction='sum')
optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=wd)
iterations_per_epoch = int(math.ceil(dataset_sizes['train'] // bs))
scheduler = CyclicLR(optimizer, cosine(t_max=iterations_per_epoch * 2, eta_min=lr/10))

for epoch in range(n_epochs):
    stats = {'epoch': epoch + 1, 'total': n_epochs}

    for phase in ('train', 'val'):
        training = phase == 'train'
        running_loss = 0.0
        n_batches = 0
        batch_num = 0
        for batch in batches(*datasets[phase], shuffle=training, bs=bs):
            x_batch, y_batch = [b.to(device) for b in batch]
            optimizer.zero_grad()
            # compute gradients only during 'train' phase
            with torch.set_grad_enabled(training):
                outputs = net(x_batch[:, 0], x_batch[:, 1], minmax)
                loss = criterion(outputs, y_batch)

                # don't update weights and rates when in 'val' phase
                if training:
                    scheduler.step()
                    loss.backward()
                    optimizer.step()
                    lr_history.extend(scheduler.get_lr())

            running_loss += loss.item()

        epoch_loss = running_loss / dataset_sizes[phase]
        stats[phase] = epoch_loss

        # early stopping: save weights of the best model so far
        if phase == 'val':
            if epoch_loss < best_loss:
                print('loss improvement on epoch: %d' % (epoch + 1))
                best_loss = epoch_loss
                best_weights = copy.deepcopy(net.state_dict())
                no_improvements = 0
            else:
                no_improvements += 1

    history.append(stats)
    print('[{epoch:03d}/{total:03d}] train: {train:.4f} - val: {val:.4f}'.format(**stats))
    if no_improvements >= patience:
        print('early stopping after epoch {epoch:03d}'.format(**stats))
        break

## **7. 모델 평가하기**

In [ ]:
# Generator와 Discriminator loss 시각화
plt.figure(figsize=(10, 5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses, label="G")
plt.plot(D_losses, label="D")
plt.xlabel("iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# 마지막에 생성된 이미지 출력
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Generated Images")
plt.imshow(np.transpose(img_list[-1], (1, 2, 0)))
plt.show()

## **8. 실험 수정**

n_Factor = 128, hidden_layer는 hidden = [128,64,32] 로 수정

In [ ]:
# 데이터셋 분할, random state는 위에서 사전 정의 완료

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
datasets = {'train': (X_train, y_train), 'val': (X_valid, y_valid)}
dataset_sizes = {'train': len(X_train), 'val': len(X_valid)}

In [ ]:
minmax = ratings.rating.min(), ratings.rating.max()
minmax

In [ ]:
EmbeddingNet(n, m,
    n_factors=128,
    hidden=[256,128,64],
    dropouts=[0.3,0.2,0.1]
)

In [ ]:
lr = 1e-3
wd = 1e-5
bs = 2000
n_epochs = 200
patience = 10
no_improvements = 0
best_loss = np.inf
best_weights = None
history = []
lr_history = []

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

net.to(device)
criterion = nn.MSELoss(reduction='sum')
optimizer = optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
iterations_per_epoch = int(math.ceil(dataset_sizes['train'] // bs))
scheduler = CyclicLR(optimizer, cosine(t_max=iterations_per_epoch * 2, eta_min=lr/10))

for epoch in range(n_epochs):
    stats = {'epoch': epoch + 1, 'total': n_epochs}

    for phase in ('train', 'val'):
        training = phase == 'train'
        running_loss = 0.0
        n_batches = 0
        batch_num = 0
        for batch in batches(*datasets[phase], shuffle=training, bs=bs):
            x_batch, y_batch = [b.to(device) for b in batch]
            optimizer.zero_grad()
            # compute gradients only during 'train' phase
            with torch.set_grad_enabled(training):
                outputs = net(x_batch[:, 0], x_batch[:, 1], minmax)
                loss = criterion(outputs, y_batch)

                # don't update weights and rates when in 'val' phase
                if training:
                    scheduler.step()
                    loss.backward()
                    optimizer.step()
                    lr_history.extend(scheduler.get_lr())

            running_loss += loss.item()

        epoch_loss = running_loss / dataset_sizes[phase]
        stats[phase] = epoch_loss

        # early stopping: save weights of the best model so far
        if phase == 'val':
            if epoch_loss < best_loss:
                print('loss improvement on epoch: %d' % (epoch + 1))
                best_loss = epoch_loss
                best_weights = copy.deepcopy(net.state_dict())
                no_improvements = 0
            else:
                no_improvements += 1

    history.append(stats)
    print('[{epoch:03d}/{total:03d}] train: {train:.4f} - val: {val:.4f}'.format(**stats))
    if no_improvements >= patience:
        print('early stopping after epoch {epoch:03d}'.format(**stats))
        break

In [ ]:
ax = pd.DataFrame(history).drop(columns='total').plot(x='epoch')


In [ ]:
plt.plot(lr_history[:2*iterations_per_epoch])

In [ ]:
net.load_state_dict(best_weights)

In [ ]:
groud_truth, predictions = [], []

with torch.no_grad():
    for batch in batches(*datasets['val'], shuffle=False, bs=bs):
        x_batch, y_batch = [b.to(device) for b in batch]
        outputs = net(x_batch[:, 0], x_batch[:, 1], minmax)
        groud_truth.extend(y_batch.tolist())
        predictions.extend(outputs.tolist())

groud_truth = np.asarray(groud_truth).ravel()
predictions = np.asarray(predictions).ravel()

final_loss = np.sqrt(np.mean((np.array(predictions) - np.array(groud_truth))**2))
print(f'Final RMSE: {final_loss:.4f}')

In [ ]:
np.array(predictions)